# 6. Ring All-Reduce：怎样用 Reduce-Scatter 与 All-Gather 得到完全一致的平均梯度？

## 面试回答主线

Ring All-Reduce 将梯度向量切成 world_size 个 chunk，先沿环进行 world_size-1 步 reduce-scatter，使每个 rank 拥有一个完整求和 chunk，再用同样步数 all-gather 传播所有 chunk。每一步每个 rank 只与相邻 rank 通信，网络总量与参数服务器同阶，但避免中心节点热点。面试时我会用真实命名参数梯度实现分块、发送快照、累加、拥有者索引和重组，并与参数服务器 reference 逐元素比较。求平均只能在完整 sum 得到后除一次；若每一跳都除 world_size，会反复缩小早到梯度。真实系统还需要处理 NCCL stream、bucket、混合精度、拓扑和 rank 故障。这个小实验重点验证数学与通信轨迹，而不是声称复刻 GPU 带宽。

## 1. 真实案例：四个数据并行 worker 的八项命名梯度

八个值对应 embedding、attention、MLP 与输出层参数块的梯度摘要。四个 rank 分别处理退款、订单、RAG 和安全样本，因此本地梯度不同；目标是得到逐元素全局平均。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示梯度与通信轨迹
import torch  # 导入 PyTorch 执行真实张量分块、累加与误差比较
torch.set_num_threads(1)  # 限制教学实验线程数以保持输出稳定
parameter_names = ["embed.refund", "embed.order", "attn.q", "attn.k", "mlp.up", "mlp.down", "norm", "lm_head"]  # 定义八个真实模型参数块名称
worker_gradients = [torch.tensor([0.8, -0.2, 0.4, 0.1, 1.2, -0.5, 0.3, 0.7], dtype=torch.float32), torch.tensor([0.6, 0.1, 0.5, -0.2, 1.0, -0.4, 0.2, 0.9], dtype=torch.float32), torch.tensor([0.9, -0.1, 0.3, 0.2, 1.1, -0.6, 0.4, 0.8], dtype=torch.float32), torch.tensor([0.7, 0.0, 0.6, -0.1, 0.9, -0.3, 0.1, 1.0], dtype=torch.float32)]  # 定义四个 worker 从不同业务 batch 得到的八项梯度
worker_batches = ["退款政策 batch", "订单状态 batch", "RAG 引用 batch", "密钥安全 batch"]  # 记录每个 rank 的真实训练数据语义
world_size = len(worker_gradients)  # 从梯度列表取得数据并行 rank 数量
preview = []  # 收集每个 worker 的命名梯度输入
for rank, gradient in enumerate(worker_gradients):  # 遍历四个数据并行 worker
    preview.append({"rank": rank, "本地batch": worker_batches[rank], "梯度": {name: round(float(value), 3) for name, value in zip(parameter_names, gradient)}})  # 将向量元素映射回参数块名称
print("Ring All-Reduce 本地梯度预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示四个 rank 的八项真实梯度差异

Ring All-Reduce 本地梯度预览：
[{'rank': 0,
  '本地batch': '退款政策 batch',
  '梯度': {'embed.refund': 0.8,
         'embed.order': -0.2,
         'attn.q': 0.4,
         'attn.k': 0.1,
         'mlp.up': 1.2,
         'mlp.down': -0.5,
         'norm': 0.3,
         'lm_head': 0.7}},
 {'rank': 1,
  '本地batch': '订单状态 batch',
  '梯度': {'embed.refund': 0.6,
         'embed.order': 0.1,
         'attn.q': 0.5,
         'attn.k': -0.2,
         'mlp.up': 1.0,
         'mlp.down': -0.4,
         'norm': 0.2,
         'lm_head': 0.9}},
 {'rank': 2,
  '本地batch': 'RAG 引用 batch',
  '梯度': {'embed.refund': 0.9,
         'embed.order': -0.1,
         'attn.q': 0.3,
         'attn.k': 0.2,
         'mlp.up': 1.1,
         'mlp.down': -0.6,
         'norm': 0.4,
         'lm_head': 0.8}},
 {'rank': 3,
  '本地batch': '密钥安全 batch',
  '梯度': {'embed.refund': 0.7,
         'embed.order': 0.0,
         'attn.q': 0.6,
         'attn.k': -0.1,
         'mlp.up': 0.9,
         'mlp.down': -0.3,
         'norm': 0.1,
         

## 2. Baseline（基线）：参数服务器集中求和再广播

参数服务器 gather 四个完整向量、求平均并广播。它给出权威数值 reference，但 rank 0 或独立 server 承担所有流量热点。下面计算中心节点收发元素数与逐参数平均梯度。

In [2]:
stacked_gradients = torch.stack(worker_gradients, dim=0)  # 把四个 worker 的本地梯度堆叠到中心节点
reference_sum = stacked_gradients.sum(dim=0)  # 在参数服务器上逐元素求全局梯度和
reference_average = reference_sum / world_size  # 在完整求和后一次性得到全局平均梯度
vector_length = worker_gradients[0].numel()  # 读取每个 rank 的梯度元素总数
server_hotspot_elements = 2 * (world_size - 1) * vector_length  # 计算中心 gather 与 broadcast 的收发热点元素数
baseline_table = [{"参数块": name, "四rank求和": round(float(total), 4), "全局平均": round(float(average), 4)} for name, total, average in zip(parameter_names, reference_sum, reference_average)]  # 构造逐参数权威结果
print("参数服务器基线的全局梯度：")  # 标注当前输出属于数值 reference
pprint(baseline_table, sort_dicts=False)  # 展示八项参数梯度的真实求和与平均
print({"中心节点热点元素": server_hotspot_elements, "每个非中心rank发送完整向量": vector_length})  # 展示参数服务器的通信集中度

参数服务器基线的全局梯度：
[{'参数块': 'embed.refund', '四rank求和': 3.0, '全局平均': 0.75},
 {'参数块': 'embed.order', '四rank求和': -0.2, '全局平均': -0.05},
 {'参数块': 'attn.q', '四rank求和': 1.8, '全局平均': 0.45},
 {'参数块': 'attn.k', '四rank求和': 0.0, '全局平均': 0.0},
 {'参数块': 'mlp.up', '四rank求和': 4.2, '全局平均': 1.05},
 {'参数块': 'mlp.down', '四rank求和': -1.8, '全局平均': -0.45},
 {'参数块': 'norm', '四rank求和': 1.0, '全局平均': 0.25},
 {'参数块': 'lm_head', '四rank求和': 3.4, '全局平均': 0.85}]
{'中心节点热点元素': 48, '每个非中心rank发送完整向量': 8}


## 3. 手写 Reduce-Scatter：每步先快照发送，再在下一 rank 累加

八个元素按四个 rank 切成四个二元素 chunk。第 s 步 rank r 发送 `(r-s) mod N` 的 chunk 给下一 rank；所有 send 必须先 clone 成快照，再统一应用，避免通信与更新顺序耦合。三步后每个 rank 拥有一个完整求和 chunk。

In [3]:
def reduce_scatter_ring(gradients):  # 手写 Ring All-Reduce 的 reduce-scatter 阶段
    size = len(gradients)  # 读取参与通信的 rank 数量
    chunks = [list(torch.chunk(gradient.clone(), size)) for gradient in gradients]  # 将每个本地梯度等分为四个 chunk
    ledger = []  # 保存每一步相邻发送和累加轨迹
    for step in range(size - 1):  # Ring reduce-scatter 需要执行 N-1 轮
        sends = []  # 暂存本轮所有 rank 的发送快照
        for rank in range(size):  # 让每个 rank 同时准备一个待发送 chunk
            chunk_index = (rank - step) % size  # 计算标准 ring 调度下本轮发送的 chunk 索引
            destination = (rank + 1) % size  # 选择环上的下一个 rank 作为接收方
            sends.append({"from": rank, "to": destination, "chunk": chunk_index, "data": chunks[rank][chunk_index].clone()})  # clone 数据模拟并发通信快照
        for message in sends:  # 在全部发送快照完成后统一处理接收
            destination = message["to"]  # 读取当前消息的目标 rank
            chunk_index = message["chunk"]  # 读取需要累加的参数分块索引
            chunks[destination][chunk_index] = chunks[destination][chunk_index] + message["data"]  # 将上一个 rank 的部分和累加到本地 chunk
            ledger.append({"phase": "reduce-scatter", "step": step, "from": message["from"], "to": destination, "chunk": chunk_index, "payload": [round(float(value), 3) for value in message["data"]]})  # 记录本轮通信内容
    owned_indices = [(rank - (size - 1)) % size for rank in range(size)]  # 计算三步后每个 rank 拥有的完整求和 chunk
    owned_chunks = [chunks[rank][owned_indices[rank]].clone() for rank in range(size)]  # 提取四个 rank 各自负责的归约结果
    return owned_indices, owned_chunks, ledger  # 返回拥有者映射、求和 chunks 与通信轨迹
owned_indices, owned_chunks, reduce_ledger = reduce_scatter_ring(worker_gradients)  # 对四个真实本地梯度执行 reduce-scatter
ownership = [{"rank": rank, "拥有chunk": owned_indices[rank], "参数块": parameter_names[owned_indices[rank] * 2:(owned_indices[rank] + 1) * 2], "归约值": [round(float(value), 4) for value in owned_chunks[rank]]} for rank in range(world_size)]  # 将 chunk 结果映射回命名参数
print("Reduce-Scatter 三步后的 chunk 拥有关系：")  # 输出核心算法中间量标题
pprint(ownership, sort_dicts=False)  # 展示每个 rank 得到的完整局部参数和
print("前八条环通信消息：")  # 输出部分逐步通信轨迹标题
pprint(reduce_ledger[:8], sort_dicts=False)  # 展示相邻 rank、chunk 与真实 payload

Reduce-Scatter 三步后的 chunk 拥有关系：
[{'rank': 0, '拥有chunk': 1, '参数块': ['attn.q', 'attn.k'], '归约值': [1.8, 0.0]},
 {'rank': 1, '拥有chunk': 2, '参数块': ['mlp.up', 'mlp.down'], '归约值': [4.2, -1.8]},
 {'rank': 2, '拥有chunk': 3, '参数块': ['norm', 'lm_head'], '归约值': [1.0, 3.4]},
 {'rank': 3,
  '拥有chunk': 0,
  '参数块': ['embed.refund', 'embed.order'],
  '归约值': [3.0, -0.2]}]
前八条环通信消息：
[{'phase': 'reduce-scatter',
  'step': 0,
  'from': 0,
  'to': 1,
  'chunk': 0,
  'payload': [0.8, -0.2]},
 {'phase': 'reduce-scatter',
  'step': 0,
  'from': 1,
  'to': 2,
  'chunk': 1,
  'payload': [0.5, -0.2]},
 {'phase': 'reduce-scatter',
  'step': 0,
  'from': 2,
  'to': 3,
  'chunk': 2,
  'payload': [1.1, -0.6]},
 {'phase': 'reduce-scatter',
  'step': 0,
  'from': 3,
  'to': 0,
  'chunk': 3,
  'payload': [0.1, 1.0]},
 {'phase': 'reduce-scatter',
  'step': 1,
  'from': 0,
  'to': 1,
  'chunk': 3,
  'payload': [0.4, 1.7]},
 {'phase': 'reduce-scatter',
  'step': 1,
  'from': 1,
  'to': 2,
  'chunk': 0,
  'payload': [1.4, -0

## 4. 手写 All-Gather：沿环传播完整求和 chunk 并重组向量

每个 rank 从自己拥有的一个完整 chunk 开始，连续三步把最近收到的 chunk 传给下一 rank。结束后四个 rank 都收齐四块，按 chunk index 拼回完整 sum，最后只除一次 world_size。

In [4]:
def all_gather_ring(owned_chunk_indices, reduced_chunks):  # 手写 Ring All-Reduce 的 all-gather 阶段
    size = len(reduced_chunks)  # 读取参与 all-gather 的 rank 数量
    gathered = [{owned_chunk_indices[rank]: reduced_chunks[rank].clone()} for rank in range(size)]  # 初始化每个 rank 已拥有的完整求和 chunk
    outgoing = [(owned_chunk_indices[rank], reduced_chunks[rank].clone()) for rank in range(size)]  # 把每个 rank 自有 chunk 设为首轮待发送消息
    ledger = []  # 保存三轮 all-gather 相邻传播轨迹
    for step in range(size - 1):  # Ring all-gather 同样执行 N-1 轮
        next_outgoing = [None] * size  # 初始化下一轮每个 rank 需要继续转发的消息
        for rank, (chunk_index, data) in enumerate(outgoing):  # 让每个 rank 发送最近拥有的新 chunk
            destination = (rank + 1) % size  # 选择环上下一个 rank 作为接收方
            gathered[destination][chunk_index] = data.clone()  # 在接收 rank 保存完整求和 chunk
            next_outgoing[destination] = (chunk_index, data.clone())  # 下一轮继续沿环转发刚收到的 chunk
            ledger.append({"phase": "all-gather", "step": step, "from": rank, "to": destination, "chunk": chunk_index, "payload": [round(float(value), 3) for value in data]})  # 记录真实传播内容
        outgoing = next_outgoing  # 推进到下一轮需要发送的 chunk
    full_sums = [torch.cat([gathered[rank][chunk_index] for chunk_index in range(size)]) for rank in range(size)]  # 按索引顺序重组每个 rank 的完整梯度和
    averages = [full_sum / size for full_sum in full_sums]  # 在完整归约后统一除一次 world_size
    return full_sums, averages, ledger  # 返回各 rank 完整和、平均值与 all-gather 轨迹
ring_sums, ring_averages, gather_ledger = all_gather_ring(owned_indices, owned_chunks)  # 对 reduce-scatter 结果执行 all-gather
rank_summary = [{"rank": rank, "完整sum": [round(float(value), 4) for value in ring_sums[rank]], "平均梯度": [round(float(value), 4) for value in ring_averages[rank]], "与reference最大误差": float(torch.max(torch.abs(ring_averages[rank] - reference_average)))} for rank in range(world_size)]  # 汇总每个 rank 的最终数值一致性
print("All-Gather 后四个 rank 的结果：")  # 输出完整 ring 算法结果标题
pprint(rank_summary, sort_dicts=False)  # 展示每个 rank 都获得相同全局平均梯度

All-Gather 后四个 rank 的结果：
[{'rank': 0,
  '完整sum': [3.0, -0.2, 1.8, 0.0, 4.2, -1.8, 1.0, 3.4],
  '平均梯度': [0.75, -0.05, 0.45, 0.0, 1.05, -0.45, 0.25, 0.85],
  '与reference最大误差': 1.1920928955078125e-07},
 {'rank': 1,
  '完整sum': [3.0, -0.2, 1.8, 0.0, 4.2, -1.8, 1.0, 3.4],
  '平均梯度': [0.75, -0.05, 0.45, 0.0, 1.05, -0.45, 0.25, 0.85],
  '与reference最大误差': 1.1920928955078125e-07},
 {'rank': 2,
  '完整sum': [3.0, -0.2, 1.8, 0.0, 4.2, -1.8, 1.0, 3.4],
  '平均梯度': [0.75, -0.05, 0.45, 0.0, 1.05, -0.45, 0.25, 0.85],
  '与reference最大误差': 1.1920928955078125e-07},
 {'rank': 3,
  '完整sum': [3.0, -0.2, 1.8, 0.0, 4.2, -1.8, 1.0, 3.4],
  '平均梯度': [0.75, -0.05, 0.45, 0.0, 1.05, -0.45, 0.25, 0.85],
  '与reference最大误差': 1.1920928955078125e-07}]


## 5. 结果解读：总通信同阶，但热点被均匀摊开

每个 rank 在两个阶段各发送 `N-1` 个二元素 chunk，共 12 个元素；四个 rank 总发送 48 个元素，与参数服务器 gather+broadcast 的全网总量相同，但没有单个中心承受 48。逐参数结果与 reference 完全一致。

In [5]:
chunk_length = vector_length // world_size  # 计算每个环消息携带的梯度元素数量
ring_per_rank_elements = 2 * (world_size - 1) * chunk_length  # 计算一个 rank 在两阶段发送的总元素数
ring_total_elements = ring_per_rank_elements * world_size  # 汇总整个环上的发送元素数
parameter_rows = []  # 构造逐参数数值一致性结果表
for index, name in enumerate(parameter_names):  # 遍历八个真实参数块
    values = [float(average[index]) for average in ring_averages]  # 读取四个 rank 对当前参数的最终平均
    parameter_rows.append({"参数块": name, "参数服务器": round(float(reference_average[index]), 4), "四rank Ring": [round(value, 4) for value in values], "全部一致": all(abs(value - float(reference_average[index])) < 1e-6 for value in values)})  # 保存逐元素一致性证据
print("逐参数 All-Reduce 结果：")  # 输出结果解读标题
pprint(parameter_rows, sort_dicts=False)  # 展示四个 rank 与参数服务器 reference 的逐项对照
print({"参数服务器中心热点元素": server_hotspot_elements, "Ring每rank发送元素": ring_per_rank_elements, "Ring全网发送元素": ring_total_elements, "中心热点被消除": ring_per_rank_elements < server_hotspot_elements})  # 比较总量与单点通信压力

逐参数 All-Reduce 结果：
[{'参数块': 'embed.refund',
  '参数服务器': 0.75,
  '四rank Ring': [0.75, 0.75, 0.75, 0.75],
  '全部一致': True},
 {'参数块': 'embed.order',
  '参数服务器': -0.05,
  '四rank Ring': [-0.05, -0.05, -0.05, -0.05],
  '全部一致': True},
 {'参数块': 'attn.q',
  '参数服务器': 0.45,
  '四rank Ring': [0.45, 0.45, 0.45, 0.45],
  '全部一致': True},
 {'参数块': 'attn.k',
  '参数服务器': 0.0,
  '四rank Ring': [0.0, 0.0, 0.0, 0.0],
  '全部一致': True},
 {'参数块': 'mlp.up',
  '参数服务器': 1.05,
  '四rank Ring': [1.05, 1.05, 1.05, 1.05],
  '全部一致': True},
 {'参数块': 'mlp.down',
  '参数服务器': -0.45,
  '四rank Ring': [-0.45, -0.45, -0.45, -0.45],
  '全部一致': True},
 {'参数块': 'norm',
  '参数服务器': 0.25,
  '四rank Ring': [0.25, 0.25, 0.25, 0.25],
  '全部一致': True},
 {'参数块': 'lm_head',
  '参数服务器': 0.85,
  '四rank Ring': [0.85, 0.85, 0.85, 0.85],
  '全部一致': True}]
{'参数服务器中心热点元素': 48, 'Ring每rank发送元素': 12, 'Ring全网发送元素': 48, '中心热点被消除': True}


## 6. 失败案例与修正：每一跳都除 world_size 导致反复缩小

平均操作不是结合律下的逐跳 `(a+b)/N`。如果每接收一块就除以 4，早到梯度会被重复缩小，最终结果偏离真实平均；修正是 reduce-scatter 只做 sum，all-gather 完成后统一除一次。

In [6]:
def broken_early_average(gradients):  # 实现每一跳错误除 world_size 的失败 reduce-scatter
    size = len(gradients)  # 读取参与通信的 rank 数量
    chunks = [list(torch.chunk(gradient.clone(), size)) for gradient in gradients]  # 将每个本地梯度等分为四块
    for step in range(size - 1):  # 仍按标准环执行三轮通信
        sends = [((rank + 1) % size, (rank - step) % size, chunks[rank][(rank - step) % size].clone()) for rank in range(size)]  # 先快照本轮全部发送消息
        for destination, chunk_index, data in sends:  # 逐条处理相邻 rank 接收
            chunks[destination][chunk_index] = (chunks[destination][chunk_index] + data) / size  # 错误地在每一跳都除以 world_size
    owned = [(rank - (size - 1)) % size for rank in range(size)]  # 计算错误阶段后的 chunk 拥有者
    return [chunks[rank][owned[rank]] for rank in range(size)]  # 返回数值已经被反复缩小的所谓归约块
broken_chunks = broken_early_average(worker_gradients)  # 在相同四组梯度上复现早除错误
broken_sum = torch.cat([broken_chunks[owned_indices.index(chunk_index)] for chunk_index in range(world_size)])  # 按 chunk index 重组错误结果
broken_average = broken_sum  # 错误实现已经声称每跳完成平均而不再统一除法
broken_max_error = float(torch.max(torch.abs(broken_average - reference_average)))  # 计算错误结果与权威平均的最大逐元素差
fixed_max_error = max(float(torch.max(torch.abs(average - reference_average))) for average in ring_averages)  # 计算正确 sum 后一次除法的最大误差
print({"失败_逐跳平均": [round(float(value), 4) for value in broken_average], "reference平均": [round(float(value), 4) for value in reference_average], "失败最大误差": round(broken_max_error, 6), "修正最大误差": fixed_max_error})  # 展示反复缩小与一次除法修正

{'失败_逐跳平均': [0.2531, -0.0078, 0.15, 0.0188, 0.3562, -0.1453, 0.1188, 0.2828], 'reference平均': [0.75, -0.05, 0.45, 0.0, 1.05, -0.45, 0.25, 0.85], '失败最大误差': 0.69375, '修正最大误差': 1.1920928955078125e-07}


## 7. 生产差距与最小回归检查

真实训练使用 NCCL 按 GPU/NVLink/网络拓扑选择 ring、tree 或混合算法，并把梯度按 bucket 与反向传播重叠。非整除向量需要 padding 与有效长度，FP16/BF16 累加还要考虑数值误差；任何 rank 故障都会阻塞同步 collective。下面的断言只验证本实验的八项梯度、拥有者映射、全 rank 一致、通信热点和早除失败。

In [7]:
assert len(parameter_names) >= 6 and len(worker_gradients) == world_size == 4  # 确认真实梯度元素和数据并行 rank 数量
assert sorted(owned_indices) == list(range(world_size))  # 确认 reduce-scatter 后每个 chunk 恰好有一个拥有者
assert all(row["全部一致"] for row in parameter_rows)  # 确认八项参数在四个 rank 上都等于参数服务器 reference
assert max(summary["与reference最大误差"] for summary in rank_summary) < 1e-6  # 确认完整 Ring All-Reduce 逐元素误差可忽略
assert ring_total_elements == server_hotspot_elements  # 确认两种算法全网通信量同阶且本例精确相等
assert ring_per_rank_elements < server_hotspot_elements  # 确认 Ring 将中心热点均匀分摊到各 rank
assert broken_max_error > 0.1 and fixed_max_error < 1e-6  # 确认逐跳平均真实错误而 sum 后一次除法修复
print("回归检查通过：Reduce-Scatter、All-Gather、通信分摊与平均时机均已验证。")  # 输出最终验收结论

回归检查通过：Reduce-Scatter、All-Gather、通信分摊与平均时机均已验证。
